# GigShield - Load, Merge, Clean, Scale

This notebook merges both downloaded food-delivery datasets into one training-ready dataset for the GigShield use case.

Pipeline:
1. Load two source CSV files
2. Harmonize column names and formats
3. Keep only use-case relevant columns
4. Build risk features (`weather_risk`, `traffic_risk`, `severity_score`, `claim_triggered`)
5. Create standardized (`_z`) and normalized (`_norm`) feature columns
6. Save merged outputs to `../data/processed/`


In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)

DATASET_A_PATH = Path('food_dataset/Food_Delivery_Times.csv')
DATASET_B_PATH = Path('food_dataset_1/Food_Time new.csv')
OUTPUT_DIR = Path('../data/processed')

OUTPUT_CLEAN = OUTPUT_DIR / 'gigshield_merged_clean.csv'
OUTPUT_MODEL = OUTPUT_DIR / 'gigshield_merged_model_ready.csv'


In [4]:
def parse_multidot_number(value):
    """
    Handles strings like '3.816.666.667' by converting them to '38.16666667'.
    If value has normal numeric format, parse directly.
    """
    if pd.isna(value):
        return np.nan

    s = str(value).strip()
    if s == '':
        return np.nan

    parts = s.split('.')
    if len(parts) <= 2:
        try:
            return float(s)
        except ValueError:
            return np.nan

    digits = ''.join(parts)
    if not digits.isdigit() or len(digits) < 3:
        return np.nan

    corrected = digits[:2] + '.' + digits[2:]
    try:
        return float(corrected)
    except ValueError:
        return np.nan


def canonical_weather(raw_weather: str) -> str:
    if pd.isna(raw_weather):
        return 'Clear'

    w = str(raw_weather).strip().lower()
    if any(k in w for k in ['rain', 'drizzle', 'shower', 'storm', 'thunder']):
        return 'Rain'
    if any(k in w for k in ['snow', 'sleet', 'blizzard']):
        return 'Snow'
    if any(k in w for k in ['fog', 'mist', 'haze', 'smoke']):
        return 'Fog'
    if any(k in w for k in ['cloud', 'overcast', 'broken', 'scattered', 'few']):
        return 'Cloudy'
    if 'wind' in w:
        return 'Windy'
    return 'Clear'


def canonical_traffic(raw_traffic: str) -> str:
    if pd.isna(raw_traffic):
        return 'Medium'

    t = str(raw_traffic).strip().lower()
    if 'very high' in t:
        return 'Very High'
    if 'high' in t:
        return 'High'
    if 'very low' in t:
        return 'Very Low'
    if 'low' in t:
        return 'Low'
    return 'Medium'


def canonical_vehicle(raw_vehicle: str) -> str:
    if pd.isna(raw_vehicle):
        return 'scooter'

    v = str(raw_vehicle).strip().lower().replace('-', '_').replace(' ', '_')
    mapping = {
        'bike': 'cycle',
        'bicycle': 'cycle',
        'scooter': 'scooter',
        'electric_scooter': 'ev',
        'motorcycle': 'motorcycle',
        'car': 'car',
    }
    return mapping.get(v, 'scooter')


In [5]:
weather_defaults = {
    'Clear': {'temperature_c': 29.0, 'humidity_pct': 45.0, 'precipitation_mm': 0.0},
    'Cloudy': {'temperature_c': 26.0, 'humidity_pct': 60.0, 'precipitation_mm': 0.2},
    'Fog': {'temperature_c': 22.0, 'humidity_pct': 80.0, 'precipitation_mm': 0.1},
    'Rain': {'temperature_c': 24.0, 'humidity_pct': 85.0, 'precipitation_mm': 2.5},
    'Snow': {'temperature_c': 0.0, 'humidity_pct': 88.0, 'precipitation_mm': 1.5},
    'Windy': {'temperature_c': 23.0, 'humidity_pct': 55.0, 'precipitation_mm': 0.0},
}

traffic_to_ratio = {
    'Very Low': 0.10,
    'Low': 0.25,
    'Medium': 0.50,
    'High': 0.75,
    'Very High': 0.90,
}

vehicle_context = {
    'cycle': 1.5,
    'scooter': 1.3,
    'motorcycle': 1.2,
    'ev': 1.0,
    'car': 0.8,
}


def map_weather_risk(temp: float, precipitation: float, humidity: float, weather_condition: str) -> float:
    score = 0.0

    if temp >= 40:
        score += 30
    elif temp >= 35:
        score += 20
    elif temp <= 0:
        score += 25
    elif temp <= 5:
        score += 15

    score += min(max(precipitation, 0) * 5, 25)
    score += min(max(humidity - 50, 0) / 5, 10)

    condition_boost = {
        'Rain': 8,
        'Snow': 18,
        'Fog': 10,
        'Windy': 6,
        'Cloudy': 2,
        'Clear': 0,
    }
    score += condition_boost.get(weather_condition, 0)

    return float(np.clip(score, 0, 100))


def map_traffic_risk(traffic_level: str) -> float:
    congestion_ratio = traffic_to_ratio.get(traffic_level, 0.5)
    return float(np.clip(congestion_ratio * 100, 0, 100))


def compute_severity_score(weather_risk: float, traffic_risk: float, context_mult: float, news_risk: float = 0.0) -> float:
    score = (0.35 * weather_risk) + (0.35 * traffic_risk) + (0.20 * news_risk) + (0.10 * (context_mult * 50))
    return float(np.clip(round(score, 2), 0, 100))


In [6]:
# Load both source datasets
df_a = pd.read_csv(DATASET_A_PATH)
df_b = pd.read_csv(DATASET_B_PATH)

print('Dataset A shape:', df_a.shape)
print('Dataset B shape:', df_b.shape)
display(df_a.head(2))
display(df_b.head(2))


Dataset A shape: (1000, 9)
Dataset B shape: (10000, 17)


,Order_ID,Distance_km,Weather,Traffic_Level,Time_of_Day,Vehicle_Type,Preparation_Time_min,Courier_Experience_yrs,Delivery_Time_min
0,522,7.93,Windy,Low,Afternoon,Scooter,12,1.0,43
1,738,16.42,Clear,Medium,Evening,Bike,20,2.0,84


,Traffic_Level,ID,Delivery_person_ID,weather_description,Type_of_order,Type_of_vehicle,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,temperature,humidity,precipitation,Distance (km),TARGET
0,High,70A2,CHENRES12DEL01,mist,Snack,scooter,32,4.6,12.972.793,80.249.982,13.012.793,80.289.982,26.55,87.0,0.0,9.89,43.45
1,High,95B4,RANCHIRES15DEL01,clear sky,Meal,scooter,33,4.7,23.369.746,8.533.982,23.479.746,8.544.982,17.51,69.0,0.0,19.11,3.816.666.667


In [7]:
# Harmonize Dataset A (Food_Delivery_Times.csv)
a = pd.DataFrame()
a['source_dataset'] = 'food_dataset'
a['record_id'] = df_a['Order_ID'].astype(str)
a['distance_km'] = pd.to_numeric(df_a['Distance_km'], errors='coerce')
a['weather_condition'] = df_a['Weather'].map(canonical_weather)
a['traffic_level'] = df_a['Traffic_Level'].map(canonical_traffic)
a['time_of_day'] = df_a['Time_of_Day'].astype(str).str.title()
a['vehicle_type'] = df_a['Vehicle_Type'].map(canonical_vehicle)
a['preparation_time_min'] = pd.to_numeric(df_a['Preparation_Time_min'], errors='coerce')
a['courier_experience_yrs'] = pd.to_numeric(df_a['Courier_Experience_yrs'], errors='coerce')
a['delivery_time_min'] = pd.to_numeric(df_a['Delivery_Time_min'], errors='coerce')
a['worker_age'] = np.nan
a['worker_rating'] = np.nan
a['order_type'] = np.nan

a['temperature_c'] = a['weather_condition'].map(lambda w: weather_defaults[w]['temperature_c'])
a['humidity_pct'] = a['weather_condition'].map(lambda w: weather_defaults[w]['humidity_pct'])
a['precipitation_mm'] = a['weather_condition'].map(lambda w: weather_defaults[w]['precipitation_mm'])


In [8]:
# Harmonize Dataset B (Food_Time new.csv)
b = pd.DataFrame()
b['source_dataset'] = 'food_dataset_1'
b['record_id'] = df_b['ID'].astype(str)
b['distance_km'] = pd.to_numeric(df_b['Distance (km)'], errors='coerce')
b['weather_condition'] = df_b['weather_description'].map(canonical_weather)
b['traffic_level'] = df_b['Traffic_Level'].map(canonical_traffic)
b['time_of_day'] = np.nan
b['vehicle_type'] = df_b['Type_of_vehicle'].map(canonical_vehicle)
b['preparation_time_min'] = np.nan
b['courier_experience_yrs'] = np.nan
b['delivery_time_min'] = df_b['TARGET'].map(parse_multidot_number)
b['worker_age'] = pd.to_numeric(df_b['Delivery_person_Age'], errors='coerce')
b['worker_rating'] = pd.to_numeric(df_b['Delivery_person_Ratings'], errors='coerce')
b['order_type'] = df_b['Type_of_order'].astype(str).str.title()

b['temperature_c'] = pd.to_numeric(df_b['temperature'], errors='coerce')
b['humidity_pct'] = pd.to_numeric(df_b['humidity'], errors='coerce')
b['precipitation_mm'] = pd.to_numeric(df_b['precipitation'], errors='coerce')


In [9]:
# Merge + keep only use-case columns
usecase_cols = [
    'source_dataset',
    'record_id',
    'distance_km',
    'weather_condition',
    'traffic_level',
    'time_of_day',
    'vehicle_type',
    'temperature_c',
    'humidity_pct',
    'precipitation_mm',
    'preparation_time_min',
    'courier_experience_yrs',
    'worker_age',
    'worker_rating',
    'order_type',
    'delivery_time_min',
]

merged = pd.concat([a[usecase_cols], b[usecase_cols]], ignore_index=True)

# Numeric cleanup
num_cols = [
    'distance_km', 'temperature_c', 'humidity_pct', 'precipitation_mm',
    'preparation_time_min', 'courier_experience_yrs', 'worker_age',
    'worker_rating', 'delivery_time_min'
]
for c in num_cols:
    merged[c] = pd.to_numeric(merged[c], errors='coerce')

# Fill missing values with robust medians (for mixed-source compatibility)
for c in num_cols:
    merged[c] = merged[c].fillna(merged[c].median())

merged['time_of_day'] = merged['time_of_day'].fillna('Unknown')
merged['order_type'] = merged['order_type'].fillna('Unknown')
merged['weather_condition'] = merged['weather_condition'].fillna('Clear')
merged['traffic_level'] = merged['traffic_level'].fillna('Medium')
merged['vehicle_type'] = merged['vehicle_type'].fillna('scooter')

# Basic clipping
merged['humidity_pct'] = merged['humidity_pct'].clip(0, 100)
merged['precipitation_mm'] = merged['precipitation_mm'].clip(lower=0)
merged['distance_km'] = merged['distance_km'].clip(lower=0)
merged['delivery_time_min'] = merged['delivery_time_min'].clip(lower=0)

print('Merged shape:', merged.shape)
display(merged.head(5))


Merged shape: (11000, 16)


,source_dataset,record_id,distance_km,weather_condition,traffic_level,time_of_day,vehicle_type,temperature_c,humidity_pct,precipitation_mm,preparation_time_min,courier_experience_yrs,worker_age,worker_rating,order_type,delivery_time_min
0,NaN,522,7.93,Windy,Low,Afternoon,scooter,23.0,55.0,0.0,12.0,1.0,29.0,4.7,Unknown,43.0
1,NaN,738,16.42,Clear,Medium,Evening,cycle,29.0,45.0,0.0,20.0,2.0,29.0,4.7,Unknown,84.0
2,NaN,741,9.52,Fog,Low,Night,scooter,22.0,80.0,0.1,28.0,1.0,29.0,4.7,Unknown,59.0
3,NaN,661,7.44,Rain,Medium,Afternoon,scooter,24.0,85.0,2.5,5.0,1.0,29.0,4.7,Unknown,37.0
4,NaN,412,19.03,Clear,Low,Morning,cycle,29.0,45.0,0.0,16.0,5.0,29.0,4.7,Unknown,68.0


In [10]:
# Build use-case risk features
merged['congestion_ratio'] = merged['traffic_level'].map(traffic_to_ratio).fillna(0.5)
merged['weather_risk'] = merged.apply(
    lambda r: map_weather_risk(
        temp=r['temperature_c'],
        precipitation=r['precipitation_mm'],
        humidity=r['humidity_pct'],
        weather_condition=r['weather_condition'],
    ),
    axis=1,
)
merged['traffic_risk'] = merged['traffic_level'].map(lambda t: map_traffic_risk(t))
merged['news_risk'] = 0.0
merged['context_mult'] = merged['vehicle_type'].map(vehicle_context).fillna(1.0).clip(0.5, 1.5)
merged['severity_score'] = merged.apply(
    lambda r: compute_severity_score(r['weather_risk'], r['traffic_risk'], r['context_mult'], r['news_risk']),
    axis=1,
)

critical = (
    (merged['weather_risk'] >= 65) |
    (merged['traffic_risk'] >= 65) |
    (merged['news_risk'] >= 50)
)

# Practical fallback for mixed public datasets (no direct expected ETA column):
# combine severity percentile + high delivery-time proxy to avoid all-zero labels.
severity_p75 = merged['severity_score'].quantile(0.75)
delivery_p75 = merged['delivery_time_min'].quantile(0.75)

rule_claim = (merged['severity_score'] >= 70) & critical
percentile_claim = merged['severity_score'] >= severity_p75
proxy_delay_claim = merged['delivery_time_min'] >= delivery_p75

merged['claim_triggered'] = (rule_claim | (percentile_claim & proxy_delay_claim)).astype(int)

if merged['claim_triggered'].mean() < 0.15:
    merged['claim_triggered'] = percentile_claim.astype(int)

print(f"Severity p75 threshold: {severity_p75:.2f}")
print(f"Delivery-time p75 threshold: {delivery_p75:.2f}")
print(f"Claim positive rate: {merged['claim_triggered'].mean():.1%}")

band_bins = [0, 30, 60, 80, 100]
band_labels = ['Low', 'Moderate', 'High', 'Critical']
merged['risk_band'] = pd.cut(merged['severity_score'], bins=band_bins, labels=band_labels, include_lowest=True)

display(
    merged[[
        'distance_km', 'weather_condition', 'traffic_level', 'vehicle_type', 'temperature_c', 'humidity_pct',
        'precipitation_mm', 'delivery_time_min', 'weather_risk', 'traffic_risk', 'context_mult',
        'severity_score', 'claim_triggered', 'risk_band'
    ]].head(10)
)



Severity p75 threshold: 36.73
Delivery-time p75 threshold: 48.70
Claim positive rate: 17.1%


,distance_km,weather_condition,traffic_level,vehicle_type,temperature_c,humidity_pct,precipitation_mm,delivery_time_min,weather_risk,traffic_risk,context_mult,severity_score,claim_triggered,risk_band
0,7.93,Windy,Low,scooter,23.0,55.0,0.0,43.0,7.0,25.0,1.3,17.70,0,Low
1,16.42,Clear,Medium,cycle,29.0,45.0,0.0,84.0,0.0,50.0,1.5,25.00,0,Low
2,9.52,Fog,Low,scooter,22.0,80.0,0.1,59.0,16.5,25.0,1.3,21.02,0,Low
3,7.44,Rain,Medium,scooter,24.0,85.0,2.5,37.0,27.5,50.0,1.3,33.62,0,Moderate
4,19.03,Clear,Low,cycle,29.0,45.0,0.0,68.0,0.0,25.0,1.5,16.25,0,Low
5,19.40,Clear,Low,scooter,29.0,45.0,0.0,57.0,0.0,25.0,1.3,15.25,0,Low
6,9.52,Clear,Low,cycle,29.0,45.0,0.0,49.0,0.0,25.0,1.5,16.25,0,Low
7,17.39,Clear,Medium,scooter,29.0,45.0,0.0,46.0,0.0,50.0,1.3,24.00,0,Low
8,1.78,Snow,Low,car,0.0,88.0,1.5,35.0,58.1,25.0,0.8,33.09,0,Moderate
9,10.62,Fog,Low,scooter,22.0,80.0,0.1,73.0,16.5,25.0,1.3,21.02,0,Low


In [11]:
# Scaling + normalization
scale_cols = [
    'distance_km', 'temperature_c', 'humidity_pct', 'precipitation_mm',
    'preparation_time_min', 'courier_experience_yrs', 'worker_age',
    'worker_rating', 'delivery_time_min', 'congestion_ratio',
    'weather_risk', 'traffic_risk', 'context_mult', 'severity_score'
]

std_scaler = StandardScaler()
mm_scaler = MinMaxScaler()

std_values = std_scaler.fit_transform(merged[scale_cols])
mm_values = mm_scaler.fit_transform(merged[scale_cols])

for i, col in enumerate(scale_cols):
    merged[f'{col}_z'] = std_values[:, i]
    merged[f'{col}_norm'] = mm_values[:, i]

display(merged[[c for c in merged.columns if c.endswith('_z')][:6] + [c for c in merged.columns if c.endswith('_norm')][:6]].head(5))


,distance_km_z,temperature_c_z,humidity_pct_z,precipitation_mm_z,preparation_time_min_z,courier_experience_yrs_z,distance_km_norm,temperature_c_norm,humidity_pct_norm,precipitation_mm_norm,preparation_time_min_norm,courier_experience_yrs_norm
0,-0.742557,-0.005337,-0.674811,-0.204489,-2.302148,-4.538482,0.123882,0.791738,0.417910,0.00,0.291667,0.111111
1,0.332305,1.462518,-1.300959,-0.204489,1.382495,-3.393242,0.267173,0.998279,0.268657,0.00,0.625000,0.222222
2,-0.541258,-0.249980,0.890559,0.067159,5.067137,-4.538482,0.150717,0.757315,0.791045,0.04,0.958333,0.111111
3,-0.804593,0.239305,1.203633,6.586710,-5.526210,-4.538482,0.115612,0.826162,0.865672,1.00,0.000000,0.111111
4,0.662740,1.462518,-1.300959,-0.204489,-0.459827,0.042478,0.311224,0.998279,0.268657,0.00,0.458333,0.555556


In [12]:
print('=== claim_triggered distribution ===')
print(merged['claim_triggered'].value_counts())
print(merged['claim_triggered'].value_counts(normalize=True).round(3))

print('\n=== severity_score summary ===')
display(merged['severity_score'].describe())

print('\n=== risk_band distribution ===')
print(merged['risk_band'].value_counts(dropna=False))


=== claim_triggered distribution ===
claim_triggered
0    9115
1    1885
Name: count, dtype: int64
claim_triggered
0    0.829
1    0.171
Name: proportion, dtype: float64

=== severity_score summary ===


count    11000.000000
mean        29.040203
std          8.633653
min          8.500000
25%         23.500000
50%         29.170000
75%         36.730000
max         54.090000
Name: severity_score, dtype: float64


=== risk_band distribution ===
risk_band
Low         6030
Moderate    4970
High           0
Critical       0
Name: count, dtype: int64


In [13]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

clean_cols = [
    'source_dataset', 'record_id', 'distance_km', 'weather_condition', 'traffic_level', 'time_of_day',
    'vehicle_type', 'temperature_c', 'humidity_pct', 'precipitation_mm', 'preparation_time_min',
    'courier_experience_yrs', 'worker_age', 'worker_rating', 'order_type', 'delivery_time_min',
    'congestion_ratio', 'weather_risk', 'traffic_risk', 'news_risk', 'context_mult',
    'severity_score', 'claim_triggered', 'risk_band'
]

model_cols = clean_cols + [c for c in merged.columns if c.endswith('_z') or c.endswith('_norm')]

merged[clean_cols].to_csv(OUTPUT_CLEAN, index=False)
merged[model_cols].to_csv(OUTPUT_MODEL, index=False)

print('Saved clean dataset ->', OUTPUT_CLEAN.resolve())
print('Saved model-ready dataset ->', OUTPUT_MODEL.resolve())
print('Final rows:', len(merged))


Saved clean dataset -> /home/zeltrox/Documents/project/GigShield/data/processed/gigshield_merged_clean.csv
Saved model-ready dataset -> /home/zeltrox/Documents/project/GigShield/data/processed/gigshield_merged_model_ready.csv
Final rows: 11000
